In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/adamgetzkin/test-ground-truth-csv/test_ground_truth.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/target_pairs.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/train_labels.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/train.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/test.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_1.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_4.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_3.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_2.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/kaggle_evaluation/mitsui_inference_server.py
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/kaggle_ev

## Imports & Config

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import lightgbm as lgb
import polars as pl
import kaggle_evaluation.mitsui_inference_server

warnings.filterwarnings('ignore')

DATA_DIR = '/kaggle/input/competitions/mitsui-commodity-prediction-challenge/'


## Predict Function

**Split:** Train = date_id < 1600 | Val = date_id 1600–1826 | Test = date_id ≥ 1827

Training happens on the first `predict()` call. Pair-specific features from
`target_pairs.csv` are built for each target. Pair feature cache is also stored
for use in train/val prediction generation after training completes.


In [ ]:
# Global state
_models       = {}
_feature_cols = {}  # per-target dict
_target_cols  = []

def _to_pd(x):
    return x.to_pandas() if hasattr(x, 'to_pandas') else x

def _build_base_features(price_df, base_cols):
    """Market-wide return, momentum and cross-sectional features."""
    price_df = price_df.copy().ffill().bfill()
    feat = pd.DataFrame({'date_id': price_df['date_id'].values})
    ret1_cols, ret5_cols, ret20_cols = [], [], []
    for col in base_cols:
        prices = price_df[col]
        r1, r5, r20 = f'{col}_ret1', f'{col}_ret5', f'{col}_ret20'
        feat[r1]  = prices.pct_change(1)
        feat[r5]  = prices.pct_change(5)
        feat[r20] = prices.pct_change(20)
        feat[f'{col}_vol5']  = prices.pct_change(1).rolling(5).std()
        feat[f'{col}_mom20'] = prices.pct_change(1).rolling(20).mean()
        ret1_cols.append(r1); ret5_cols.append(r5); ret20_cols.append(r20)
    cs_mean_1  = feat[ret1_cols].mean(axis=1)
    cs_std_1   = feat[ret1_cols].std(axis=1)  + 1e-8
    cs_mean_5  = feat[ret5_cols].mean(axis=1)
    cs_std_5   = feat[ret5_cols].std(axis=1)  + 1e-8
    cs_mean_20 = feat[ret20_cols].mean(axis=1)
    cs_std_20  = feat[ret20_cols].std(axis=1) + 1e-8
    rank1_df = feat[ret1_cols].rank(axis=1, pct=True)
    rank5_df = feat[ret5_cols].rank(axis=1, pct=True)
    for col in base_cols:
        feat[f'{col}_ret1_cs_z']       = (feat[f'{col}_ret1']  - cs_mean_1)  / cs_std_1
        feat[f'{col}_ret5_cs_z']       = (feat[f'{col}_ret5']  - cs_mean_5)  / cs_std_5
        feat[f'{col}_ret20_cs_z']      = (feat[f'{col}_ret20'] - cs_mean_20) / cs_std_20
        feat[f'{col}_ret1_cs_rank']    = rank1_df[f'{col}_ret1']
        feat[f'{col}_ret5_cs_rank']    = rank5_df[f'{col}_ret5']
        feat[f'{col}_ret1_cs_rank_sq'] = rank1_df[f'{col}_ret1'] ** 2
    feat = feat.replace([np.inf, -np.inf], np.nan)
    return feat

def _build_pair_features(price_df, asset_a, asset_b):
    """Features specific to a target pair (asset_a, asset_b)."""
    price_df = price_df.copy().ffill().bfill()
    feat = pd.DataFrame({'date_id': price_df['date_id'].values})
    if asset_a not in price_df.columns or asset_b not in price_df.columns:
        return feat
    pa = price_df[asset_a];  pb = price_df[asset_b]
    ra1 = pa.pct_change(1);  rb1 = pb.pct_change(1)
    ra5 = pa.pct_change(5);  rb5 = pb.pct_change(5)
    ra20= pa.pct_change(20); rb20= pb.pct_change(20)
    feat['pair_rel_ret1']   = ra1  - rb1
    feat['pair_rel_ret5']   = ra5  - rb5
    feat['pair_rel_ret20']  = ra20 - rb20
    feat['pair_spread']     = pa - pb
    feat['pair_ratio']      = pa / (pb + 1e-8)
    feat['pair_ratio_ma5']  = feat['pair_ratio'].rolling(5).mean()
    feat['pair_ratio_ma20'] = feat['pair_ratio'].rolling(20).mean()
    spread_ma20  = feat['pair_spread'].rolling(20).mean()
    spread_std20 = feat['pair_spread'].rolling(20).std() + 1e-8
    feat['pair_spread_z20'] = (feat['pair_spread'] - spread_ma20) / spread_std20
    feat['pair_rel_vol5']   = ra1.rolling(5).std() - rb1.rolling(5).std()
    feat['pair_corr20']     = ra1.rolling(20).corr(rb1)
    feat['pair_corr5']      = ra1.rolling(5).corr(rb1)
    feat = feat.replace([np.inf, -np.inf], np.nan)
    return feat

def predict(test, label_lags_1_batch, label_lags_2_batch, label_lags_3_batch, label_lags_4_batch):
    global _models, _feature_cols, _target_cols

    if not _models:
        print('First call — loading data and training models...')

        train  = pd.read_csv(DATA_DIR + 'train.csv').sort_values('date_id').reset_index(drop=True)
        labels = pd.read_csv(DATA_DIR + 'train_labels.csv').sort_values('date_id').reset_index(drop=True)
        pairs  = pd.read_csv(DATA_DIR + 'target_pairs.csv')

        _target_cols = [c for c in labels.columns if c != 'date_id']
        base_cols    = [c for c in train.columns  if c != 'date_id']

        # Pair lookup: target -> (asset_a, asset_b)
        pair_lookup = {}
        if len(pairs.columns) >= 3:
            for _, row in pairs.iterrows():
                pair_lookup[row.iloc[0]] = (row.iloc[1], row.iloc[2])
        print(f'Loaded {len(pair_lookup)} target pairs')

        print('Building base features...')
        base_feat_df   = _build_base_features(train, base_cols)
        base_feat_cols = [c for c in base_feat_df.columns if c != 'date_id']

        df_base = base_feat_df.merge(labels, on='date_id')

        # Lagged targets
        for t in _target_cols:
            df_base[f'{t}_lag1']     = df_base[t].shift(2)
            df_base[f'{t}_lag2']     = df_base[t].shift(3)
            df_base[f'{t}_lag_diff'] = df_base[f'{t}_lag1'] - df_base[f'{t}_lag2']

        # ── NEW: hardcoded splits ──────────────────────────────────────────
        train_df = df_base[df_base['date_id'] <  1600].copy()
        val_df   = df_base[(df_base['date_id'] >= 1600) & (df_base['date_id'] <= 1826)].copy()
        print(f'Train rows: {len(train_df)} (date_id < 1600)')
        print(f'Val rows:   {len(val_df)}   (date_id 1600-1826)')

        print('Training models (with pair features)...')
        for i, t in enumerate(_target_cols):
            if i % 20 == 0:
                print(f'Training model {i}/{len(_target_cols)}...')

            mask_train = ~train_df[t].isna()
            mask_val   = ~val_df[t].isna()
            if mask_train.sum() < 10:
                continue

            pair_feat_cols = []
            if t in pair_lookup:
                asset_a, asset_b = pair_lookup[t]
                pf = _build_pair_features(train, asset_a, asset_b)
                pair_feat_cols = [c for c in pf.columns if c != 'date_id']
                pf_train = pf.iloc[:len(df_base)]
                pf_vals  = pf_train.set_index(pf_train.index)
                train_pair_vals = pf.iloc[train_df.index][pair_feat_cols].values
                val_pair_vals   = pf.iloc[val_df.index][pair_feat_cols].values
                X_pair_train = pd.DataFrame(train_pair_vals, index=train_df.index, columns=pair_feat_cols)
                X_pair_val   = pd.DataFrame(val_pair_vals,   index=val_df.index,   columns=pair_feat_cols)

            lag_cols = [f'{t}_lag1', f'{t}_lag2', f'{t}_lag_diff']
            all_feat_cols = base_feat_cols + lag_cols + pair_feat_cols

            if t in pair_lookup:
                X_train = pd.concat([
                    train_df.loc[mask_train, base_feat_cols + lag_cols].reset_index(drop=True),
                    X_pair_train.loc[mask_train].reset_index(drop=True)
                ], axis=1)
                X_val = pd.concat([
                    val_df.loc[mask_val, base_feat_cols + lag_cols].reset_index(drop=True),
                    X_pair_val.loc[mask_val].reset_index(drop=True)
                ], axis=1)
            else:
                X_train = train_df.loc[mask_train, base_feat_cols + lag_cols]
                X_val   = val_df.loc[mask_val,   base_feat_cols + lag_cols]

            _feature_cols[t] = all_feat_cols

            model = lgb.LGBMRegressor(
                objective='mae',
                n_estimators=500,
                learning_rate=0.03,
                max_depth=4,
                num_leaves=20,
                min_child_samples=100,
                subsample=0.8,
                colsample_bytree=0.7,
                reg_alpha=0.1,
                reg_lambda=5.0,
                random_state=42,
                n_jobs=-1,
                verbose=-1,
            )
            model.fit(
                X_train, train_df.loc[mask_train, t],
                eval_set=[(X_val, val_df.loc[mask_val, t])],
                callbacks=[lgb.early_stopping(40, verbose=False)]
            )
            _models[t] = model

        print(f'Training complete. {len(_models)} models ready.')

        # Store everything needed for train/val prediction generation
        predict._base_cols      = base_cols
        predict._base_feat_cols = base_feat_cols
        predict._pair_lookup    = pair_lookup
        predict._train_labels   = labels
        predict._train_prices   = train[['date_id'] + base_cols].copy()
        predict._base_feat_full = base_feat_df.copy()
        predict._sorted_dates   = sorted(labels['date_id'].unique())
        predict._date_to_idx    = {d: i for i, d in enumerate(predict._sorted_dates)}

        # Pre-build pair feature cache for all training dates
        print('Building pair feature cache for train/val scoring...')
        predict._pair_feat_cache = {}
        for t in _target_cols:
            if t in pair_lookup:
                asset_a, asset_b = pair_lookup[t]
                key = (asset_a, asset_b)
                if key not in predict._pair_feat_cache:
                    predict._pair_feat_cache[key] = _build_pair_features(train, asset_a, asset_b)
        print(f'Cached {len(predict._pair_feat_cache)} unique pair feature sets.')

        predict._train = train[['date_id'] + base_cols].tail(150).copy()

    # ── INFERENCE ──────────────────────────────────────────────────────
    test_row = _to_pd(test).copy().apply(pd.to_numeric, errors='coerce')
    lag1_df  = _to_pd(label_lags_1_batch).copy().apply(pd.to_numeric, errors='coerce')
    lag2_df  = _to_pd(label_lags_2_batch).copy().apply(pd.to_numeric, errors='coerce')

    predict._train = pd.concat(
        [predict._train, test_row[['date_id'] + predict._base_cols]],
        ignore_index=True
    ).tail(150)

    base_feat     = _build_base_features(predict._train, predict._base_cols)
    base_test_row = base_feat.iloc[[-1]].reset_index(drop=True)

    preds = {}
    for t in _target_cols:
        if t not in _models:
            preds[t] = np.nan
            continue

        l1 = lag1_df[t].values[0] if t in lag1_df.columns and not lag1_df[t].isna().all() else np.nan
        l2 = lag2_df[t].values[0] if t in lag2_df.columns and not lag2_df[t].isna().all() else np.nan

        row = base_test_row.copy()
        row[f'{t}_lag1']     = l1
        row[f'{t}_lag2']     = l2
        row[f'{t}_lag_diff'] = l1 - l2 if not (np.isnan(l1) or np.isnan(l2)) else np.nan

        if t in predict._pair_lookup:
            asset_a, asset_b = predict._pair_lookup[t]
            pair_feat = _build_pair_features(predict._train, asset_a, asset_b)
            pair_row  = pair_feat.iloc[[-1]].reset_index(drop=True)
            for pc in [c for c in pair_feat.columns if c != 'date_id']:
                row[pc] = pair_row[pc].values[0]

        feat_cols = _feature_cols.get(t, [])
        preds[t]  = _models[t].predict(row.reindex(columns=feat_cols))[0]

    pred_values = np.array(list(preds.values()), dtype=float)
    valid_mask  = ~np.isnan(pred_values)
    if valid_mask.sum() > 0:
        m, s = np.mean(pred_values[valid_mask]), np.std(pred_values[valid_mask]) + 1e-8
        pred_values[valid_mask] = (pred_values[valid_mask] - m) / s

    return pd.DataFrame([{t: v for t, v in zip(_target_cols, pred_values)}])


## Run Inference Server

In [ ]:
# inference_server = kaggle_evaluation.mitsui_inference_server.MitsuiInferenceServer(predict)
# if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
#     inference_server.serve()
# else:
#     inference_server.run_local_gateway((DATA_DIR,))


## Generate Predictions on Test Set (date_id ≥ 1827)

In [ ]:
test_df = pd.read_csv(DATA_DIR + 'test.csv')
lag1    = pd.read_csv(DATA_DIR + 'lagged_test_labels/test_labels_lag_1.csv')
lag2    = pd.read_csv(DATA_DIR + 'lagged_test_labels/test_labels_lag_2.csv')
lag3    = pd.read_csv(DATA_DIR + 'lagged_test_labels/test_labels_lag_3.csv')
lag4    = pd.read_csv(DATA_DIR + 'lagged_test_labels/test_labels_lag_4.csv')

# Create empty DataFrames with the same columns to pass if a date is missing
empty_lag1 = pd.DataFrame(columns=lag1.columns)
empty_lag2 = pd.DataFrame(columns=lag2.columns)
empty_lag3 = pd.DataFrame(columns=lag3.columns)
empty_lag4 = pd.DataFrame(columns=lag4.columns)

all_preds = []
for i in range(len(test_df)):
    if i % 20 == 0:
        print(f'Predicting row {i}/{len(test_df)}...')
        
    current_date = test_df.iloc[i]['date_id']
    
    # FIX 4: Filter lag batches strictly by date_id, not by row index!
    lag1_batch = lag1[lag1['date_id'] == current_date]
    lag2_batch = lag2[lag2['date_id'] == current_date]
    lag3_batch = lag3[lag3['date_id'] == current_date]
    lag4_batch = lag4[lag4['date_id'] == current_date]
    
    # If a date is completely missing from the lag file, pass an empty DataFrame
    l1_pass = lag1_batch if not lag1_batch.empty else empty_lag1
    l2_pass = lag2_batch if not lag2_batch.empty else empty_lag2
    l3_pass = lag3_batch if not lag3_batch.empty else empty_lag3
    l4_pass = lag4_batch if not lag4_batch.empty else empty_lag4
    
    row_pred = predict(
        pl.from_pandas(test_df.iloc[[i]]),
        pl.from_pandas(l1_pass),
        pl.from_pandas(l2_pass),
        pl.from_pandas(l3_pass),
        pl.from_pandas(l4_pass),
    )
    row_pred['date_id'] = current_date
    all_preds.append(row_pred)

pred_df = pd.concat(all_preds, ignore_index=True)
print(f'Predictions shape: {pred_df.shape}')

## Score Test Predictions Against Ground Truth

In [ ]:
GROUND_TRUTH_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if 'ground_truth' in f.lower() or ('truth' in f.lower() and f.endswith('.csv')):
            GROUND_TRUTH_PATH = os.path.join(root, f)
            print('Found ground truth at:', GROUND_TRUTH_PATH)

assert GROUND_TRUTH_PATH is not None, 'Upload test_ground_truth.csv as a Kaggle dataset'

ground_truth = pd.read_csv(GROUND_TRUTH_PATH)

if _target_cols[0] in ground_truth.columns:
    gt_aligned   = ground_truth[ground_truth['date_id'].isin(pred_df['date_id'])].sort_values('date_id').reset_index(drop=True)
    pred_aligned = pred_df[pred_df['date_id'].isin(gt_aligned['date_id'])].sort_values('date_id').reset_index(drop=True)
    score_cols   = _target_cols
else:
    rename_map   = {_target_cols[i]: f'target_{i}' for i in range(len(_target_cols))}
    pred_renamed = pred_df[['date_id'] + _target_cols].rename(columns=rename_map)
    gt_aligned   = ground_truth[ground_truth['date_id'].isin(pred_renamed['date_id'])].sort_values('date_id').reset_index(drop=True)
    pred_aligned = pred_renamed[pred_renamed['date_id'].isin(gt_aligned['date_id'])].sort_values('date_id').reset_index(drop=True)
    score_cols   = [f'target_{i}' for i in range(424)]

def competition_score(pred_df, gt_df, target_cols):
    pred = pred_df[target_cols].copy()
    gt   = gt_df[target_cols].copy()
    daily_corrs = []
    
    for i in range(len(pred)):
        gt_row   = gt.iloc[i]
        pred_row = pred.iloc[i]
        
        valid = gt_row.notna()
        if valid.sum() < 2: continue
            
        gt_valid   = gt_row[valid]
        pred_valid = pred_row[valid]
        
        if gt_valid.std(ddof=0) == 0 or pred_valid.std(ddof=0) == 0: continue
            
        corr = np.corrcoef(
            pred_valid.rank(method='average'),
            gt_valid.rank(method='average')
        )[0, 1]
        daily_corrs.append(corr)
        
    daily_corrs = np.array(daily_corrs)
    sharpe = daily_corrs.mean() / daily_corrs.std(ddof=0)
    
    print(f'  Daily rank correlations: mean={daily_corrs.mean():.6f}, std={daily_corrs.std():.6f}')
    print(f'  Days evaluated: {len(daily_corrs)}')
    print(f'  Competition Score (Sharpe): {sharpe:.6f}')
    return sharpe, daily_corrs

print('Optimized LightGBM score:')
sharpe, daily = competition_score(pred_aligned, gt_aligned, score_cols)

## Generate Train & Validation Predictions for Ensembling

These predictions are generated using correctly-timed lag labels:
when predicting date `t`, only labels from dates strictly before `t` are used.
The resulting CSVs allow your team to compute a Sharpe score per model
on train and validation data, which can then be used as ensemble weights.


In [ ]:
def generate_split_predictions(date_ids, split_name):
    """
    Generate predictions for a set of date_ids (train or val).
    Uses correctly-timed lag labels: when predicting date t,
    only uses labels from dates strictly before t.
    """
    labels_by_date  = predict._train_labels.set_index('date_id')
    sorted_dates    = predict._sorted_dates
    date_to_idx     = predict._date_to_idx
    base_feat_full  = predict._base_feat_full
    pair_lookup     = predict._pair_lookup
    pair_feat_cache = predict._pair_feat_cache
    base_feat_cols  = predict._base_feat_cols

    pred_dates = sorted(date_ids)
    print(f'\nGenerating {split_name} predictions ({len(pred_dates)} dates)...')

    all_pred_rows = []
    for step, current_date in enumerate(pred_dates):
        if step % 20 == 0:
            print(f'  {step}/{len(pred_dates)}...')

        base_row = base_feat_full[base_feat_full['date_id'] == current_date].copy().reset_index(drop=True)
        if base_row.empty:
            continue

        idx = date_to_idx.get(current_date, -1)

        def get_lag(lag_offset, target):
            lag_idx = idx - lag_offset
            if lag_idx < 0:
                return np.nan
            lag_date = sorted_dates[lag_idx]
            try:
                return float(labels_by_date.loc[lag_date, target])
            except:
                return np.nan

        preds = {}
        for t in _target_cols:
            if t not in _models:
                preds[t] = np.nan
                continue

            l1 = get_lag(1, t)
            l2 = get_lag(2, t)

            row = base_row.copy()
            row[f'{t}_lag1']     = l1
            row[f'{t}_lag2']     = l2
            row[f'{t}_lag_diff'] = (l1 - l2) if not (np.isnan(l1) or np.isnan(l2)) else np.nan

            if t in pair_lookup:
                asset_a, asset_b = pair_lookup[t]
                key = (asset_a, asset_b)
                if key in pair_feat_cache:
                    pf = pair_feat_cache[key]
                    pf_row = pf[pf['date_id'] == current_date].reset_index(drop=True)
                    for pc in [c for c in pf.columns if c != 'date_id']:
                        row[pc] = pf_row[pc].values[0] if not pf_row.empty else np.nan

            feat_cols = _feature_cols.get(t, [])
            preds[t]  = _models[t].predict(row.reindex(columns=feat_cols))[0]

        # Cross-sectional normalization (same as inference)
        pred_values = np.array([preds.get(t, np.nan) for t in _target_cols], dtype=float)
        valid_mask  = ~np.isnan(pred_values)
        if valid_mask.sum() > 0:
            m, s = np.mean(pred_values[valid_mask]), np.std(pred_values[valid_mask]) + 1e-8
            pred_values[valid_mask] = (pred_values[valid_mask] - m) / s

        row_dict = {t: v for t, v in zip(_target_cols, pred_values)}
        row_dict['date_id'] = current_date
        all_pred_rows.append(row_dict)

    return pd.DataFrame(all_pred_rows)


# Generate train and validation predictions
all_train_dates = predict._train_labels['date_id'].unique()
train_date_ids  = [d for d in all_train_dates if d <  1600]
val_date_ids    = [d for d in all_train_dates if 1600 <= d <= 1826]

train_pred_df = generate_split_predictions(train_date_ids, 'TRAIN')
val_pred_df   = generate_split_predictions(val_date_ids,   'VALIDATION')

print(f'\nTrain predictions shape:      {train_pred_df.shape}')
print(f'Validation predictions shape: {val_pred_df.shape}')


## Score Train & Validation Predictions

In [ ]:
# Load training ground truth (train_labels.csv IS the ground truth for train/val)
train_labels_gt = predict._train_labels.copy()

def score_split(pred_df, gt_df, split_name):
    """Score predictions against ground truth for a given split."""
    # Align on date_id
    gt_aln   = gt_df[gt_df['date_id'].isin(pred_df['date_id'])].sort_values('date_id').reset_index(drop=True)
    pred_aln = pred_df[pred_df['date_id'].isin(gt_aln['date_id'])].sort_values('date_id').reset_index(drop=True)

    # Use whichever column naming matches
    if _target_cols[0] in gt_aln.columns:
        use_cols = _target_cols
    else:
        rename_map = {_target_cols[i]: f'target_{i}' for i in range(len(_target_cols))}
        pred_aln = pred_aln[['date_id'] + _target_cols].rename(columns=rename_map)
        use_cols = [f'target_{i}' for i in range(424)]

    print(f'\n{split_name} Score ({len(gt_aln)} days):')
    sharpe, daily_corrs = competition_score(pred_aln, gt_aln, use_cols)
    return sharpe, daily_corrs

train_sharpe, train_corrs = score_split(train_pred_df, train_labels_gt, 'TRAIN')
val_sharpe,   val_corrs   = score_split(val_pred_df,   train_labels_gt, 'VALIDATION')


## Save All CSVs & Models

In [ ]:
import joblib

# ── Test predictions ───────────────────────────────────────────────────
pred_df.to_csv('lgbm_preds_test.csv', index=False)
print(f'Saved test predictions:       lgbm_preds_test.csv       ({len(pred_df)} rows)')

# ── Train predictions ──────────────────────────────────────────────────
train_pred_df.to_csv('lgbm_preds_train.csv', index=False)
print(f'Saved train predictions:      lgbm_preds_train.csv      ({len(train_pred_df)} rows)')

# ── Validation predictions ─────────────────────────────────────────────
val_pred_df.to_csv('lgbm_preds_val.csv', index=False)
print(f'Saved validation predictions: lgbm_preds_val.csv        ({len(val_pred_df)} rows)')

# ── Renamed versions (target_0..target_423) for easy ensembling ────────
rename_map = {_target_cols[i]: f'target_{i}' for i in range(len(_target_cols))}
pred_df[['date_id'] + _target_cols].rename(columns=rename_map).to_csv('lgbm_preds_test_renamed.csv',  index=False)
train_pred_df[['date_id'] + _target_cols].rename(columns=rename_map).to_csv('lgbm_preds_train_renamed.csv', index=False)
val_pred_df[['date_id'] + _target_cols].rename(columns=rename_map).to_csv('lgbm_preds_val_renamed.csv',  index=False)
print('Saved renamed versions (target_0..target_423) for ensembling.')

# ── Sharpe summary for ensemble weighting ──────────────────────────────
summary = pd.DataFrame([{
    'model': 'lgbm_attempt_6',
    'train_sharpe': train_sharpe,
    'val_sharpe':   val_sharpe,
}])
summary.to_csv('lgbm_sharpe_summary.csv', index=False)
print(f'\nSharpe Summary:')
print(summary.to_string(index=False))

# ── Trained models ─────────────────────────────────────────────────────
joblib.dump({'models': _models, 'feature_cols': _feature_cols}, 'lgbm_attempt_6_models.joblib')
print('\nSaved trained models: lgbm_attempt_6_models.joblib')
